In [6]:
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset
import numpy as np
import h5py
import os
import time
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import gc

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
data_path = r'C:\Users\yonda\Downloads\MLP Baseline\data'
print("Using device:", device)

Using device: cuda


In [7]:
X_train = np.load(os.path.join(data_path, 'X_train.npy'))
y_train = np.load(os.path.join(data_path, 'y_train.npy'))
X_val = np.load(os.path.join(data_path, 'X_val.npy'))
y_val = np.load(os.path.join(data_path, 'y_val.npy'))
X_test = np.load(os.path.join(data_path, 'X_test.npy'))
y_test = np.load(os.path.join(data_path, 'y_test.npy'))

print("X_train:", X_train.shape)
print("X_val:", X_val.shape)
print("X_test:", X_test.shape)

X_train: (198036, 200, 9)
X_val: (82656, 200, 9)
X_test: (76894, 200, 9)


In [8]:
# Convert to tensors and flatten
X_train_t = torch.tensor(X_train, dtype=torch.float32).view(len(X_train), -1)
y_train_t = torch.tensor(y_train, dtype=torch.float32)
X_val_t = torch.tensor(X_val, dtype=torch.float32).view(len(X_val), -1)
y_val_t = torch.tensor(y_val, dtype=torch.float32)
X_test_t = torch.tensor(X_test, dtype=torch.float32).view(len(X_test), -1)
y_test_t = torch.tensor(y_test, dtype=torch.float32)

# Free numpy arrays from memory
del X_train, y_train, X_val, y_val, X_test, y_test
gc.collect()

# DataLoaders
train_dataset = TensorDataset(X_train_t, y_train_t)
val_dataset = TensorDataset(X_val_t, y_val_t)
test_dataset = TensorDataset(X_test_t, y_test_t)

train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=64, shuffle=False)
test_loader = DataLoader(test_dataset, batch_size=64, shuffle=False)

# MLP Model
class MLP(nn.Module):
    def __init__(self):
        super(MLP, self).__init__()
        self.network = nn.Sequential(
            nn.Linear(1800, 512),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(512, 256),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(256, 128),
            nn.ReLU(),
            nn.Linear(128, 2)
        )
    
    def forward(self, x):
        return self.network(x)

model = MLP().to(device)
print("Total parameters:", sum(p.numel() for p in model.parameters()))
print("Input size:", X_train_t.shape)

Total parameters: 1086594
Input size: torch.Size([198036, 1800])


In [9]:
criterion = nn.MSELoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

patience = 5
best_val_loss = float('inf')
epochs_no_improve = 0
best_model_state = None

train_losses = []
val_losses = []

start_time = time.time()
max_epochs = 50

for epoch in range(max_epochs):
    model.train()
    train_loss = 0
    for X_batch, y_batch in train_loader:
        X_batch, y_batch = X_batch.to(device), y_batch.to(device)
        optimizer.zero_grad()
        output = model(X_batch)
        loss = criterion(output, y_batch)
        loss.backward()
        optimizer.step()
        train_loss += loss.item()
    train_loss /= len(train_loader)
    
    model.eval()
    val_loss = 0
    with torch.no_grad():
        for X_batch, y_batch in val_loader:
            X_batch, y_batch = X_batch.to(device), y_batch.to(device)
            output = model(X_batch)
            loss = criterion(output, y_batch)
            val_loss += loss.item()
    val_loss /= len(val_loader)
    
    train_losses.append(train_loss)
    val_losses.append(val_loss)
    
    print(f"Epoch {epoch+1}/{max_epochs} | Train Loss: {train_loss:.6f} | Val Loss: {val_loss:.6f}")
    
    if val_loss < best_val_loss:
        best_val_loss = val_loss
        epochs_no_improve = 0
        best_model_state = model.state_dict().copy()
    else:
        epochs_no_improve += 1
        if epochs_no_improve >= patience:
            print(f"Early stopping triggered at epoch {epoch+1}")
            break

training_time = time.time() - start_time
print(f"\nTraining complete. Total time: {training_time:.2f} seconds ({training_time/60:.2f} minutes)")
print(f"Best validation loss: {best_val_loss:.6f}")

# Load best model and save everything immediately
model.load_state_dict(best_model_state)
torch.save(model.state_dict(), os.path.join(data_path, 'mlp_model.pth'))
np.save(os.path.join(data_path, 'train_losses.npy'), np.array(train_losses))
np.save(os.path.join(data_path, 'val_losses.npy'), np.array(val_losses))
print("Model and losses saved.")

Epoch 1/50 | Train Loss: 0.455238 | Val Loss: 0.429538
Epoch 2/50 | Train Loss: 0.423488 | Val Loss: 0.418621
Epoch 3/50 | Train Loss: 0.409731 | Val Loss: 0.408460
Epoch 4/50 | Train Loss: 0.401398 | Val Loss: 0.403091
Epoch 5/50 | Train Loss: 0.394599 | Val Loss: 0.398586
Epoch 6/50 | Train Loss: 0.388691 | Val Loss: 0.396238
Epoch 7/50 | Train Loss: 0.382736 | Val Loss: 0.392130
Epoch 8/50 | Train Loss: 0.378559 | Val Loss: 0.393659
Epoch 9/50 | Train Loss: 0.373387 | Val Loss: 0.390898
Epoch 10/50 | Train Loss: 0.368117 | Val Loss: 0.382554
Epoch 11/50 | Train Loss: 0.365931 | Val Loss: 0.380056
Epoch 12/50 | Train Loss: 0.363325 | Val Loss: 0.380855
Epoch 13/50 | Train Loss: 0.361522 | Val Loss: 0.377555
Epoch 14/50 | Train Loss: 0.360479 | Val Loss: 0.377226
Epoch 15/50 | Train Loss: 0.356040 | Val Loss: 0.379508
Epoch 16/50 | Train Loss: 0.355076 | Val Loss: 0.372876
Epoch 17/50 | Train Loss: 0.351997 | Val Loss: 0.376181
Epoch 18/50 | Train Loss: 0.350126 | Val Loss: 0.374580
E

In [10]:
def circular_mae_deg(pred_deg, true_deg):
    diff = np.abs(pred_deg - true_deg)
    diff = np.minimum(diff, 360 - diff)
    return np.mean(diff)

def circular_rmse_deg(pred_deg, true_deg):
    diff = np.abs(pred_deg - true_deg)
    diff = np.minimum(diff, 360 - diff)
    return np.sqrt(np.mean(diff ** 2))

def sincos_to_angle(y):
    return np.degrees(np.arctan2(y[:, 0], y[:, 1]))

model.eval()
all_preds = []
all_true = []

with torch.no_grad():
    for X_batch, y_batch in test_loader:
        X_batch = X_batch.to(device)
        output = model(X_batch).cpu().numpy()
        all_preds.append(output)
        all_true.append(y_batch.numpy())

all_preds = np.concatenate(all_preds, axis=0)
all_true = np.concatenate(all_true, axis=0)

pred_deg = sincos_to_angle(all_preds)
true_deg = sincos_to_angle(all_true)

# Save immediately
np.save(os.path.join(data_path, 'pred_deg.npy'), pred_deg)
np.save(os.path.join(data_path, 'true_deg.npy'), true_deg)

mae = circular_mae_deg(pred_deg, true_deg)
rmse = circular_rmse_deg(pred_deg, true_deg)

print(f"Test MAE: {mae:.4f} degrees")
print(f"Test RMSE: {rmse:.4f} degrees")
print(f"Training time: {training_time:.2f} seconds ({training_time/60:.2f} minutes)")
print("Predictions saved.")

Test MAE: 68.7472 degrees
Test RMSE: 85.3652 degrees
Training time: 167.33 seconds (2.79 minutes)
Predictions saved.


In [11]:
test_folder_path = os.path.join(data_path, 'unseen_subjects_test_set')
sequences = sorted(os.listdir(test_folder_path))

seq_lengths = []
for seq in sequences:
    seq_path = os.path.join(test_folder_path, seq, 'data.hdf5')
    if not os.path.exists(seq_path):
        continue
    with h5py.File(seq_path, 'r') as f:
        length = f['synced']['acce'].shape[0]
    windows = (length - 200) // 50
    seq_lengths.append(windows)

heading_changes = np.abs(np.diff(true_deg, prepend=true_deg[0]))
heading_changes = np.minimum(heading_changes, 360 - heading_changes)

straight_mask = heading_changes < 10
sharp_mask = heading_changes > 45

window_counts = np.array(seq_lengths)
median_len = np.median(window_counts)

short_mask = np.zeros(len(true_deg), dtype=bool)
long_mask = np.zeros(len(true_deg), dtype=bool)

idx = 0
for wc in window_counts:
    if wc <= median_len:
        short_mask[idx:idx+wc] = True
    else:
        long_mask[idx:idx+wc] = True
    idx += wc

def get_scenario_mae(pred_deg, true_deg, mask):
    return circular_mae_deg(pred_deg[mask], true_deg[mask])

straight_mae = get_scenario_mae(pred_deg, true_deg, straight_mask)
sharp_mae = get_scenario_mae(pred_deg, true_deg, sharp_mask)
short_mae = get_scenario_mae(pred_deg, true_deg, short_mask)
long_mae = get_scenario_mae(pred_deg, true_deg, long_mask)

print(f"Straight walking MAE: {straight_mae:.4f}°")
print(f"Sharp turns MAE:      {sharp_mae:.4f}°")
print(f"Short trajectories MAE: {short_mae:.4f}°")
print(f"Long trajectories MAE:  {long_mae:.4f}°")

# Save scenario results
np.save(os.path.join(data_path, 'scenario_results.npy'), 
        np.array([straight_mae, sharp_mae, short_mae, long_mae]))
print("Scenario results saved.")

Straight walking MAE: 62.6268°
Sharp turns MAE:      88.7315°
Short trajectories MAE: 65.6208°
Long trajectories MAE:  70.8040°
Scenario results saved.


In [ ]:
# Clear GPU memory before plotting
torch.cuda.empty_cache()
gc.collect()

# Loss curve
fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(train_losses, label='Train Loss')
ax.plot(val_losses, label='Val Loss')
ax.set_xlabel('Epoch')
ax.set_ylabel('MSE Loss')
ax.set_title('MLP Training vs Validation Loss')
ax.legend()
ax.grid(True)
fig.tight_layout()
fig.savefig(os.path.join(data_path, 'mlp_loss_curve.png'), dpi=150)
plt.close(fig)
del fig, ax
gc.collect()
print("Loss curve saved.")

# Trajectory plot
fig, ax = plt.subplots(figsize=(12, 4))
ax.plot(true_deg[:500], label='Ground Truth', linewidth=0.8)
ax.plot(pred_deg[:500], label='MLP Prediction', linewidth=0.8)
ax.set_xlabel('Window Index')
ax.set_ylabel('Heading (degrees)')
ax.set_title('MLP Predicted vs Ground Truth Heading (First 500 Windows)')
ax.legend()
ax.grid(True)
fig.tight_layout()
fig.savefig(os.path.join(data_path, 'mlp_trajectory.png'), dpi=150)
plt.close(fig)
del fig, ax
gc.collect()
print("Trajectory plot saved.")